# [재실행 전용 · 2026-10-04 회계 수정] crypto_baselines_colab.ipynb

원본 `crypto_baselines_colab.ipynb`에서 **이번 재실행에 필요한 셀만** 뽑은 노트북이다. 학습 셀(`.learn`)은 빠져 있고, 저장된 모델을 불러와 평가만 한다.
위에서부터 **순서대로 전부 실행**하면 된다. 결과 CSV는 원본과 같은 이름으로 `data/crypto/`에 저장된다(실행 전 백업 셀 먼저).

결과: 표 2 고전 베이스라인.


In [ ]:
import shutil, glob, os
from google.colab import drive
drive.mount('/content/drive')
src = '/content/drive/MyDrive/졸업프로젝트/data/crypto'
dst = src + '/_pre_accounting_fix_20261004'
if not os.path.exists(dst):
    os.makedirs(dst)
    for p in glob.glob(src + '/*.csv') + glob.glob(src + '/*.png'):
        shutil.copy2(p, dst)
print('백업 폴더:', dst, len(os.listdir(dst)), '개')


# 베이스라인 확장 — 등가중 이외의 고전 전략과 비교 (`paper/GAP_ANALYSIS.md` §2.1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow scikit-learn


## 1. 설정

In [ ]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from sklearn.covariance import LedoitWolf

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

RL_EMB_H5 = CRYPTO_DIR / 'crypto_rl_embeddings.h5'

EMB_DIM    = 64
TC         = 0.001
BUCKET_SEC = 1800

RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'
VALID_START, VALID_END = '2025-10-01', '2025-12-31'

TOP_K = 10
ALPHA = 0.01
BETA  = -30
DEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'

# --- 베이스라인 하이퍼파라미터 ---
MOM_LOOKBACK   = 48 * 7    # 모멘텀: 트레일링 7일
MOM_REBALANCE  = 48        # 1일마다 리밸런싱
MVO_LOOKBACK   = 48 * 90   # MVO: 트레일링 90일
MVO_REBALANCE  = 48 * 7    # 1주일마다 리밸런싱
RP_LOOKBACK    = 48 * 90   # 리스크 패리티(역변동성): 트레일링 90일 — MVO와 동일 창
RP_REBALANCE   = 48 * 7    # 1주일마다 리밸런싱 — MVO와 동일 주기

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음'
assert DEPLOYED_AGENT_PATH.exists(), f'배포 중인 에이전트 없음: {DEPLOYED_AGENT_PATH}'
print('경로 확인 완료')


## 2. 데이터 로드 (crypto_hrl_earnhft_colab.ipynb §1~2와 동일)

In [ ]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]
    embs_all    = f['embeddings'][:]

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

emb_index  = {}
sym_ts_set = {}
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


In [ ]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9
    y     = df['RetTarget_6b'].values
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


## 3. CryptoPortfolioEnv + SparseGatedCryptoPortfolioEnv (배포 모델 평가용, §3/§13-A와 동일)

In [ ]:
# [2026-10-04 회계 수정] 가격 변동 후 실제 보유 비중(drift)·거래 가능 종목 마스크.
# 이전 버전은 수익 반영 뒤에도 직전 '목표' 비중을 보유 비중으로 저장해, 가격 변동으로 틀어진 비중을
# 되돌리는 재조정 거래(등가중 포함)의 비용이 0으로 계산됐다.
def _drift_weights(w, rets):
    """수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr). 합<1이면 나머지는 현금(수익률 0)으로 남는다."""
    w = np.asarray(w, dtype=np.float64)
    g = 1.0 + float(np.dot(w, rets))
    if g <= 1e-12:
        return w.astype(np.float32)
    return (w * (1.0 + np.asarray(rets, dtype=np.float64)) / g).astype(np.float32)


def _mask_untradable(w, tradable, renorm=True):
    """수익률 데이터가 없는(상장 전·상폐 후) 종목 비중을 0으로. renorm=False면 빠진 몫은 현금으로 남는다."""
    w = np.where(tradable, np.asarray(w, dtype=np.float64), 0.0)
    s = w.sum()
    return (w / s if renorm and s > 1e-12 else w).astype(np.float32)


class CryptoPortfolioEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)

    def _asset_ret(self, bucket):
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _tradable(self, bucket):
        """해당 버킷에 수익률 데이터가 있는(거래 가능한) 종목 마스크"""
        return np.array([(sym, bucket) in self.ret_index for sym in self.symbols])

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = _mask_untradable(np.ones(self.n_assets, dtype=np.float32),
                                          self._tradable(self.buckets[0]))  # 거래 가능 종목 등가중에서 시작
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action  = np.exp(action - action.max())
        action  = action / action.sum()
        action  = _mask_untradable(action, self._tradable(self.buckets[self.t]))  # 거래 불가 종목 제외

        turnover = float(np.abs(action - self.weights).sum())
        tc_cost  = self.tc * turnover

        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(action, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        asset_logits = np.where(self._tradable(self.buckets[self.t]), asset_logits, -np.inf)  # 거래 불가 종목 제외
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        target_w = _mask_untradable(target_w, self._tradable(self.buckets[self.t]), renorm=False)  # 빠진 몫은 현금
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = _drift_weights(target_w, asset_rets)  # 가격 변동 후 실제 보유 비중
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


## 4. 버킷 분할 + 성과 계산 함수

In [ ]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
valid_buckets = [b for b in train_buckets if VALID_START <= bucket_to_date(b) <= VALID_END]
test_buckets  = [b for b in all_buckets if RL_TEST_START  <= bucket_to_date(b) <= RL_TEST_END]

print(f'valid 기간: {bucket_to_ts(valid_buckets[0])} ~ {bucket_to_ts(valid_buckets[-1])} ({len(valid_buckets):,}스텝)')
print(f'test 기간: {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)


def compute_perf(values, steps_per_year=48 * 365):
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def perf_row(values, turnovers, tc=TC):
    perf = compute_perf(values)
    t = np.asarray(turnovers, dtype=np.float64)
    perf['평균 턴오버/스텝'] = t.mean()
    perf['누적 비용 드래그'] = 1 - np.prod(1 - tc * t)
    return perf


## 5. 수익률 행렬 구성 — 고전 베이스라인은 임베딩 없이 이 행렬만으로 계산

In [ ]:
def build_return_matrix(buckets, symbols, ret_index):
    T, N = len(buckets), len(symbols)
    R = np.zeros((T, N), dtype=np.float64)
    for j, sym in enumerate(symbols):
        for i, b in enumerate(buckets):
            R[i, j] = ret_index.get((sym, b), 0.0)
    return R


def build_tradable_matrix(buckets, symbols, ret_index):
    """[2026-10-04] (T×N) 해당 버킷에 수익률 데이터가 있는 종목 = 거래 가능"""
    return np.array([[(sym, b) in ret_index for sym in symbols] for b in buckets])


R_valid = build_return_matrix(valid_buckets, symbols, ret_index)
R_test  = build_return_matrix(test_buckets,  symbols, ret_index)
TR_valid = build_tradable_matrix(valid_buckets, symbols, ret_index)
TR_test  = build_tradable_matrix(test_buckets,  symbols, ret_index)
print('수익률 행렬:', R_valid.shape, R_test.shape)
print(f'거래 불가 셀 비율 — valid {1 - TR_valid.mean():.2%}, test {1 - TR_test.mean():.2%}')


## 6. 베이스라인 정의 — Buy&Hold / 모멘텀 top-K / MVO(Ledoit-Wolf)

In [ ]:
def _drift_np(w, rets):
    """[2026-10-04 회계 수정] 수익 반영 뒤 실제 보유 비중 w_i(1+r_i)/(1+wᵀr)."""
    g = 1.0 + float(np.dot(w, rets))
    return w * (1.0 + rets) / g if g > 1e-12 else w


def backtest_target_weights(weights_matrix, returns_matrix, tc=TC, tradable=None, trade_mask=None):
    """목표 비중 백테스트 — RL 환경(step)과 동일한 타이밍 컨벤션(스텝 t의 비중으로 스텝 t의 수익)과
    비용 모델(L1 턴오버 × tc).
    [2026-10-04 회계 수정] 턴오버는 가격 변동 후 실제 보유 비중(drift) 대비로 계산한다(이전 버전은 직전
    목표 비중 대비라 등가중 재조정 비용이 0이었다). tradable(T×N bool)이면 거래 불가 종목을 0으로 두고
    재정규화, trade_mask(T bool)가 False인 스텝은 거래 없이 보유 비중을 그대로 유지한다(None=매 스텝 재조정)."""
    T, N = returns_matrix.shape
    values = [1.0]
    turnovers = []
    held = np.ones(N) / N if tradable is None else tradable[0] / max(tradable[0].sum(), 1)
    for t in range(T):
        if trade_mask is None or trade_mask[t] or (tradable is not None and (held[~tradable[t]] > 0).any()):
            w = np.asarray(weights_matrix[t], dtype=np.float64)
            if tradable is not None:
                w = np.where(tradable[t], w, 0.0)
                w = w / w.sum() if w.sum() > 1e-12 else held
        else:
            w = held
        turnover = float(np.abs(w - held).sum())
        tc_cost  = tc * turnover
        port_ret = float((w * returns_matrix[t]).sum())
        values.append(values[-1] * (1 + port_ret - tc_cost))
        turnovers.append(turnover)
        held = _drift_np(w, returns_matrix[t])
    return np.array(values), turnovers


def rebalance_mask(T, every):
    """every 스텝마다(0 포함) 거래하는 마스크 — 모멘텀·MVO·리스크패리티의 리밸런싱 주기와 동일."""
    return np.arange(T) % every == 0


def buy_and_hold_backtest(returns_matrix, tradable=None):
    """t=0에 등가중으로 1회 매수 후 리밸런싱 없음 — 비중은 가격에 따라 자연 표류, 리밸런싱 비용 0.
    tradable이 주어지면 t=0에 거래 가능한 종목만 매수한다(이후 상장 종목은 편입하지 않음)."""
    N = returns_matrix.shape[1]
    w = np.ones(N) / N if tradable is None else tradable[0] / max(tradable[0].sum(), 1)
    values = [1.0]
    port_val = 1.0
    for t in range(len(returns_matrix)):
        port_ret = float((w * returns_matrix[t]).sum())
        port_val *= (1 + port_ret)
        values.append(port_val)
        w = w * (1 + returns_matrix[t])
        s = w.sum()
        if s > 1e-12:
            w = w / s
    return np.array(values), [0.0] * len(returns_matrix)


def momentum_weights_matrix(returns_matrix, lookback=MOM_LOOKBACK, top_k=TOP_K, rebalance=MOM_REBALANCE):
    """매 rebalance스텝마다 트레일링 lookback버킷 누적수익 상위 top_k 종목을 동일비중 보유."""
    T, N = returns_matrix.shape
    cum = np.cumprod(1 + returns_matrix, axis=0)
    W = np.zeros((T, N))
    current_w = np.ones(N) / N   # 워밍업 구간(lookback 이전)은 등가중
    for t in range(T):
        if t % rebalance == 0 and t >= lookback:
            start = t - lookback
            trailing_ret = cum[t - 1] / cum[start] - 1
            top_idx = np.argsort(trailing_ret)[-top_k:]
            current_w = np.zeros(N)
            current_w[top_idx] = 1.0 / top_k
        W[t] = current_w
    return W


def mvo_weights_matrix(returns_matrix, lookback=MVO_LOOKBACK, rebalance=MVO_REBALANCE):
    """트레일링 lookback버킷 수익률의 Ledoit-Wolf 축소추정 공분산으로 공매도 없는 글로벌 최소분산 포트폴리오
    (progress_report_final.md ETF+BL 트랙과 동일한 공분산 추정 방법)."""
    T, N = returns_matrix.shape
    W = np.zeros((T, N))
    current_w = np.ones(N) / N
    for t in range(T):
        if t % rebalance == 0 and t >= lookback:
            window = returns_matrix[t - lookback:t]
            try:
                cov = LedoitWolf().fit(window).covariance_
                inv_cov = np.linalg.pinv(cov)
                raw_w = inv_cov @ np.ones(N)
                raw_w = np.clip(raw_w, 0, None)
                if raw_w.sum() > 1e-8:
                    current_w = raw_w / raw_w.sum()
            except np.linalg.LinAlgError:
                pass
        W[t] = current_w
    return W


def risk_parity_weights_matrix(returns_matrix, lookback=RP_LOOKBACK, rebalance=RP_REBALANCE):
    """역변동성(naive risk parity) — 트레일링 lookback버킷 표준편차의 역수에 비례해 배분한다.
    MVO(공분산 전체)와 달리 자산 간 상관관계는 무시하고 각 자산의 변동성만 동일화하는
    표준 단순화 버전(고상관 자산군에서 전체 공분산 역행렬이 불안정해지는 MVO의 실패 모드를
    피하기 위한 더 견고한 대안으로 흔히 쓰인다)."""
    T, N = returns_matrix.shape
    W = np.zeros((T, N))
    current_w = np.ones(N) / N
    for t in range(T):
        if t % rebalance == 0 and t >= lookback:
            window = returns_matrix[t - lookback:t]
            vol = window.std(axis=0)
            inv_vol = np.where(vol > 1e-8, 1.0 / vol, 0.0)
            if inv_vol.sum() > 1e-8:
                current_w = inv_vol / inv_vol.sum()
        W[t] = current_w
    return W


print('베이스라인 함수 정의 완료')


## 7. 배포 모델(β=-30) 평가 함수 (crypto/inference/policy_v4.py와 동일 로직)

In [ ]:
def run_beta_smoothed(model, alpha, buckets, top_k=TOP_K):
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history), list(env.turnover_history)


deployed_model = PPO.load(str(DEPLOYED_AGENT_PATH))
print('배포 모델 로드 완료')


## 8. 전체 비교 — valid / test

In [ ]:
%%time
print('=== 베이스라인 비교 — valid vs test ===\n')

rows_valid, rows_test = {}, {}
curves_valid, curves_test = {}, {}

def add_row(label, values, turnovers, split):
    row = perf_row(values, turnovers)
    if split == 'valid':
        rows_valid[label] = row
        curves_valid[label] = values
    else:
        rows_test[label] = row
        curves_test[label] = values

# 등가중 (=매 스텝 거래 가능 종목 1/N으로 재조정 — 가격 변동으로 틀어진 비중을 되돌리는 비용 포함)
ew_w_valid = np.tile(np.ones(N_ASSETS) / N_ASSETS, (len(valid_buckets), 1))
ew_w_test  = np.tile(np.ones(N_ASSETS) / N_ASSETS, (len(test_buckets), 1))
v, t = backtest_target_weights(ew_w_valid, R_valid, tradable=TR_valid); add_row('등가중 벤치마크', v, t, 'valid')
v, t = backtest_target_weights(ew_w_test,  R_test,  tradable=TR_test);  add_row('등가중 벤치마크', v, t, 'test')

# Buy & Hold
v, t = buy_and_hold_backtest(R_valid, TR_valid); add_row('Buy & Hold(무회전)', v, t, 'valid')
v, t = buy_and_hold_backtest(R_test, TR_test);  add_row('Buy & Hold(무회전)', v, t, 'test')

# 모멘텀 top-K
mom_w_valid = momentum_weights_matrix(R_valid)
mom_w_test  = momentum_weights_matrix(R_test)
v, t = backtest_target_weights(mom_w_valid, R_valid, tradable=TR_valid, trade_mask=rebalance_mask(len(R_valid), MOM_REBALANCE)); add_row(f'모멘텀 top-{TOP_K}', v, t, 'valid')
v, t = backtest_target_weights(mom_w_test,  R_test,  tradable=TR_test,  trade_mask=rebalance_mask(len(R_test), MOM_REBALANCE));  add_row(f'모멘텀 top-{TOP_K}', v, t, 'test')

# MVO(Ledoit-Wolf 글로벌 최소분산)
print('MVO 계산 중 (Ledoit-Wolf, 시간 다소 소요)...')
mvo_w_valid = mvo_weights_matrix(R_valid)
mvo_w_test  = mvo_weights_matrix(R_test)
v, t = backtest_target_weights(mvo_w_valid, R_valid, tradable=TR_valid, trade_mask=rebalance_mask(len(R_valid), MVO_REBALANCE)); add_row('MVO(Ledoit-Wolf 최소분산)', v, t, 'valid')
v, t = backtest_target_weights(mvo_w_test,  R_test,  tradable=TR_test,  trade_mask=rebalance_mask(len(R_test), MVO_REBALANCE));  add_row('MVO(Ledoit-Wolf 최소분산)', v, t, 'test')

# 리스크 패리티(역변동성)
print('리스크 패리티 계산 중...')
rp_w_valid = risk_parity_weights_matrix(R_valid)
rp_w_test  = risk_parity_weights_matrix(R_test)
v, t = backtest_target_weights(rp_w_valid, R_valid, tradable=TR_valid, trade_mask=rebalance_mask(len(R_valid), RP_REBALANCE)); add_row('리스크 패리티(역변동성)', v, t, 'valid')
v, t = backtest_target_weights(rp_w_test,  R_test,  tradable=TR_test,  trade_mask=rebalance_mask(len(R_test), RP_REBALANCE));  add_row('리스크 패리티(역변동성)', v, t, 'test')

# 배포 모델 (β=-30 + α=0.01)
print('배포 모델(β=-30) 평가 중...')
v, t = run_beta_smoothed(deployed_model, ALPHA, valid_buckets); add_row('β=-30 배포 모델(RL)', v, t, 'valid')
v, t = run_beta_smoothed(deployed_model, ALPHA, test_buckets);  add_row('β=-30 배포 모델(RL)', v, t, 'test')

valid_df = pd.DataFrame(rows_valid).T
test_df  = pd.DataFrame(rows_test).T

print('\n--- valid(2025 Q4) ---')
print(valid_df.to_string(float_format=lambda x: f'{x:.4f}'))
print('\n--- test(2026~) ---')
print(test_df.to_string(float_format=lambda x: f'{x:.4f}'))

valid_df.to_csv(CRYPTO_DIR / 'crypto_baselines_valid.csv', encoding='utf-8-sig')
test_df.to_csv(CRYPTO_DIR / 'crypto_baselines_test.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_baselines_valid.csv, crypto_baselines_test.csv')


## 9. 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for label, curve in curves_valid.items():
    lw = 2.2 if 'β=-30' in label else 1.4
    axes[0].plot(curve, label=label, lw=lw)
axes[0].set_title('valid(2025 Q4) — 베이스라인 비교')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for label, curve in curves_test.items():
    lw = 2.2 if 'β=-30' in label else 1.4
    axes[1].plot(curve, label=label, lw=lw)
axes[1].set_title('test(2026~) — 베이스라인 비교')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CRYPTO_DIR / 'crypto_baselines_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. 판정 기준

## 11. valid→test 낙차 비교 — "raw 수익률 1등"이 가장 신뢰할 만한 전략인가?

In [ ]:
gap_rows = {}
for label in rows_valid:
    v_sh = valid_df.loc[label, 'Sharpe']
    t_sh = test_df.loc[label, 'Sharpe']
    gap_rows[label] = {'valid Sharpe': v_sh, 'test Sharpe': t_sh, '낙차(|test-valid|)': abs(t_sh - v_sh)}

gap_df = pd.DataFrame(gap_rows).T.sort_values('낙차(|test-valid|)')
print('=== valid→test Sharpe 낙차 비교 (작을수록 강건) ===\n')
print(gap_df.to_string(float_format=lambda x: f'{x:.4f}'))
gap_df.to_csv(CRYPTO_DIR / 'crypto_baselines_gap_comparison.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_baselines_gap_comparison.csv')


### 해석